# 02 — Feature Analysis

**Purpose:** Understand the RFM features before clustering.  
**Questions we answer here:**
- How are the features distributed? Are there extreme outliers?
- Which features correlate with each other?
- What do the quintile scores look like?
- Are there data issues that could skew the clustering?

**Tables used:** `analytics.int_rfm_features`, `analytics.int_rfm_scores`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT = 'adg-internal-tech-sandbox'
client = bigquery.Client(project=PROJECT)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT}')

## 1. Feature summary statistics
Mean, std, median, min, max for each RFM feature.

In [ ]:
features = ['val_trns', 'nr_trns', 'lst_trns_days', 'avg_val',
            'active_months', 'active_destinations', 'active_nav_categories',
            'NR_TRNS_WEEKEND', 'NR_TRNS_WEEK']

stats_query = ' UNION ALL '.join([
    f"""SELECT '{f}' AS feature,
        ROUND(AVG({f}), 2) AS mean,
        ROUND(STDDEV({f}), 2) AS std,
        APPROX_QUANTILES({f}, 4)[OFFSET(2)] AS median,
        ROUND(MIN({f}), 2) AS min_val,
        ROUND(MAX({f}), 2) AS max_val,
        ROUND(APPROX_QUANTILES({f}, 4)[OFFSET(1)], 2) AS q25,
        ROUND(APPROX_QUANTILES({f}, 4)[OFFSET(3)], 2) AS q75
    FROM `{PROJECT}.analytics.int_rfm_features`""" for f in features
])

stats = q(stats_query)
stats['iqr'] = stats['q75'] - stats['q25']
stats

## 2. Feature distributions
Histograms of the 9 features used in clustering.

In [ ]:
# Pull a sample for visualization (full dataset can be huge)
sample = q(f"""
    SELECT val_trns, nr_trns, lst_trns_days, avg_val,
           active_months, active_destinations, active_nav_categories,
           NR_TRNS_WEEKEND, NR_TRNS_WEEK
    FROM `{PROJECT}.analytics.int_rfm_features`
    WHERE RAND() < 0.1
""")

fig = make_subplots(rows=3, cols=3,
                    subplot_titles=features)

for i, f in enumerate(features):
    row, col = i // 3 + 1, i % 3 + 1
    fig.add_trace(
        go.Histogram(x=sample[f], nbinsx=50, name=f, showlegend=False),
        row=row, col=col
    )

fig.update_layout(height=800, title='Feature distributions (10% sample)')
fig.show()

## 3. Correlations between features
Which features move together? High correlation means they carry similar info.

In [ ]:
corr = q(f"""
    SELECT
        ROUND(CORR(val_trns, nr_trns), 3)              AS spend_vs_freq,
        ROUND(CORR(val_trns, lst_trns_days), 3)        AS spend_vs_recency,
        ROUND(CORR(val_trns, avg_val), 3)              AS spend_vs_avg_txn,
        ROUND(CORR(val_trns, active_destinations), 3)  AS spend_vs_merchants,
        ROUND(CORR(nr_trns, active_destinations), 3)   AS freq_vs_merchants,
        ROUND(CORR(nr_trns, lst_trns_days), 3)         AS freq_vs_recency,
        ROUND(CORR(NR_TRNS_WEEKEND, NR_TRNS_WEEK), 3) AS weekend_vs_weekday,
        ROUND(CORR(active_months, nr_trns), 3)         AS months_vs_freq
    FROM `{PROJECT}.analytics.int_rfm_features`
""")

corr.T.rename(columns={0: 'correlation'})

## 4. Outlier analysis
How many customers are more than 3 standard deviations from the mean?

In [ ]:
outliers = q(f"""
    SELECT
        COUNT(*) AS total_customers,
        COUNTIF(val_trns > (SELECT AVG(val_trns) + 3*STDDEV(val_trns) FROM `{PROJECT}.analytics.int_rfm_features`)) AS spend_outliers,
        COUNTIF(nr_trns > (SELECT AVG(nr_trns) + 3*STDDEV(nr_trns) FROM `{PROJECT}.analytics.int_rfm_features`)) AS freq_outliers,
        COUNTIF(avg_val > (SELECT AVG(avg_val) + 3*STDDEV(avg_val) FROM `{PROJECT}.analytics.int_rfm_features`)) AS avg_val_outliers
    FROM `{PROJECT}.analytics.int_rfm_features`
""")

total = outliers['total_customers'].iloc[0]
for col in ['spend_outliers', 'freq_outliers', 'avg_val_outliers']:
    n = outliers[col].iloc[0]
    print(f'{col}: {n:,} ({n/total*100:.1f}%)')

## 5. RFM score distributions
Are the quintile scores evenly distributed (they should be ~20% each)?

In [ ]:
scores = q(f"""
    SELECT
        'Recency' AS dimension, r_score AS score, COUNT(*) AS customers
    FROM `{PROJECT}.analytics.int_rfm_scores` GROUP BY r_score
    UNION ALL
    SELECT 'Frequency', f_score, COUNT(*)
    FROM `{PROJECT}.analytics.int_rfm_scores` GROUP BY f_score
    UNION ALL
    SELECT 'Monetary', m_score, COUNT(*)
    FROM `{PROJECT}.analytics.int_rfm_scores` GROUP BY m_score
    ORDER BY dimension, score
""")

fig = px.bar(scores, x='score', y='customers', color='dimension',
             barmode='group', title='RFM quintile score distributions')
fig.show()